# Healthcare KG Population Intelligence — V4
**Multi-cycle NHANES | Survey design | Missingness adjustment | Provenance-linked population observations**

**Development-only.** This notebook never accesses GitHub or modifies the baseline KG. It is based on the successfully executed **V3.1 Gemini-corrected** workflow. It keeps the original 2017–2018 estimate as a comparison, adds 2015–2016 and optional 2021–2023 cycles, and does **not** pool noncomparable periods. 2019–March 2020 pre-pandemic data are intentionally excluded from the default workflow because their special weights and combined cycle require separate handling.

**Interpretation:** Cross-sectional *screening proxies*, not confirmed chronic CKD, confirmed type 2 diabetes, or Kerendia eligibility. All confidence intervals and trend comparisons require review of CDC survey guidance. Missing-data adjustment is a sensitivity analysis, not proof that data are missing at random.

In [ ]:
%pip -q install pandas numpy scipy requests pyreadstat matplotlib

## 1 — Configuration and baseline integrity
Upload the **V3.1 outputs ZIP** (including `baseline_kg_COPY.json`) or the baseline JSON alone. The original JSON is copied byte-for-byte; no graph mutation occurs. If running locally, set `UPLOAD_INTERACTIVE=False` and copy your archive to `/content`.

In [ ]:
from pathlib import Path
import os, json, hashlib, zipfile, datetime, io, time
import numpy as np, pandas as pd, requests
from scipy.stats import t as student_t
ROOT=Path('/content/Healthcare_KG_Population_Intelligence_V4');ROOT.mkdir(parents=True,exist_ok=True)
UPLOAD_INTERACTIVE=True
INCLUDE_2021_2023=True  # change to False if CDC variable harmonization fails
if UPLOAD_INTERACTIVE:
    from google.colab import files
    uploaded=files.upload()
    incoming=Path(next(iter(uploaded)))
else:
    incoming=Path('/content/v3_1_outputs.zip')
if incoming.suffix.lower()=='.zip':
    with zipfile.ZipFile(incoming) as archive:
        names={Path(x).name:x for x in archive.namelist() if not x.endswith('/')}
        assert 'baseline_kg_COPY.json' in names, 'Missing baseline_kg_COPY.json'
        (ROOT/'baseline_kg_COPY.json').write_bytes(archive.read(names['baseline_kg_COPY.json']))
        prior_audit=next((x for x in names if 'audit' in x.lower() and x.endswith('.json')),None)
        if prior_audit:(ROOT/'v3_1_previous_audit.json').write_bytes(archive.read(names[prior_audit]))
else:
    assert incoming.name=='baseline_kg_COPY.json'
    (ROOT/'baseline_kg_COPY.json').write_bytes(incoming.read_bytes())
kg_path=ROOT/'baseline_kg_COPY.json'
KG_SHA256=hashlib.sha256(kg_path.read_bytes()).hexdigest()
print('Read-only baseline:',kg_path.name,'SHA256:',KG_SHA256)

## 2 — Download NHANES public-use data
Use CDC's 2015–2016 (`_I`), 2017–2018 (`_J`), and optional August 2021–August 2023 (`_L`) files. The notebook checks required columns and fails visibly on incompatible releases rather than silently changing definitions. Source URLs, checksums, and row counts are recorded. If CDC changes paths, update only `CYCLES` and confirm file metadata.

In [ ]:
CYCLES={
 '2015-2016':{'year':'2015','suffix':'I','period_years':2,'era':'pre-pandemic'},
 '2017-2018':{'year':'2017','suffix':'J','period_years':2,'era':'pre-pandemic'},
}
if INCLUDE_2021_2023:
 CYCLES['2021-2023']={'year':'2021','suffix':'L','period_years':2,'era':'post-pandemic'}
COMPONENTS=['DEMO','ALB_CR','BIOPRO','DIQ']
session=requests.Session(); session.headers['User-Agent']='HealthcareKG-V4-research/1.0'
manifest=[]; cycles={}; errors={}
for label,cfg in CYCLES.items():
  tables={}
  try:
    for component in COMPONENTS:
      name=f'{component}_{cfg["suffix"]}'
      url=f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{cfg["year"]}/DataFiles/{name}.xpt'
      response=session.get(url,timeout=120);response.raise_for_status()
      if len(response.content)<1000:raise ValueError('Unexpectedly small XPT: '+name)
      local=ROOT/(name+'.xpt');local.write_bytes(response.content)
      df=pd.read_sas(local,format='xport')
      if 'SEQN' not in df:raise ValueError(f'{name}: SEQN missing; check CDC file/version')
      tables[component]=df
      manifest.append({'cycle':label,'component':name,'url':url,'sha256':hashlib.sha256(response.content).hexdigest(),'n_rows':len(df),'bytes':len(response.content),'status':'OK'})
    cycles[label]=tables
    print(label,[(k,len(v)) for k,v in tables.items()])
  except Exception as exc:
    errors[label]=str(exc)
    print('ERROR:',label,str(exc),'— this cycle will NOT be included')
    if label=='2017-2018':raise
assert '2017-2018' in cycles
pd.DataFrame(manifest).to_csv(ROOT/'v4_source_manifest.csv',index=False)

## 3 — Harmonize the screening variables
The **same** 2021 race-free CKD-EPI creatinine equation and urine albumin-to-creatinine ratio are applied to each cycle. The V3.1 diabetes proxy uses `DIQ010` and `DID040`: self-reported diabetes and age at diagnosis ≥30 years. This proxy **misses younger-onset T2D** and **may include older-onset T1D**. We retain it strictly for V3.1 comparability.

**Important:** NHANES component variables, assay methods, and sampling weights may differ by cycle; a successful code run does not establish cross-cycle measurement comparability. Verify the CDC documentation before interpreting changes as epidemiological trends.

In [ ]:
REQUIRED_DEMO=['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU']
REQUIRED_LABS=['URXUMA','URXUCR','LBXSCR']
REQUIRED_DIQ=['DIQ010','DID040']
def harmonize(label,tables):
  for name,fields in [('DEMO',REQUIRED_DEMO),('ALB_CR',['URXUMA','URXUCR']),('BIOPRO',['LBXSCR']),('DIQ',REQUIRED_DIQ)]:
    absent=[f for f in fields if f not in tables[name].columns]
    if absent:raise ValueError(f'{label} {name} missing {absent}; cannot assume equivalence')
  d=tables['DEMO'][REQUIRED_DEMO].copy()
  for component in ['ALB_CR','BIOPRO','DIQ']:
    d=d.merge(tables[component],on='SEQN',how='left',validate='one_to_one')
  d=d.loc[pd.to_numeric(d.RIDAGEYR,errors='coerce')>=20].copy()
  for col in REQUIRED_DEMO+REQUIRED_LABS+REQUIRED_DIQ:
    d[col]=pd.to_numeric(d[col],errors='coerce')
  female=d.RIAGENDR.eq(2)
  k=np.where(female,.7,.9);a=np.where(female,-.241,-.302)
  r=d.LBXSCR/k
  d['egfr']=142*np.minimum(r,1)**a*np.maximum(r,1)**(-1.2)*(.9938**d.RIDAGEYR)*np.where(female,1.012,1)
  d.loc[d.LBXSCR.isna()|d.LBXSCR.le(0),'egfr']=np.nan
  d['uacr']=100*d.URXUMA/d.URXUCR
  d.loc[d.URXUMA.isna()|d.URXUCR.isna()|d.URXUCR.le(0),'uacr']=np.nan
  d['ckd_screen']=np.where(d.egfr.notna()&d.uacr.notna(),(d.egfr<60)|(d.uacr>=30),np.nan)
  d['diabetes_reported']=np.where(d.DIQ010.isin([1,2]),d.DIQ010.eq(1).astype(float),np.nan)
  d['t2d_proxy']=np.where(d.diabetes_reported.eq(0),0,np.where(d.diabetes_reported.eq(1)&d.DID040.between(1,100),(d.DID040>=30).astype(float),np.nan))
  d['joint']=np.where(d.ckd_screen.notna()&pd.notna(d.t2d_proxy),(d.ckd_screen==1)&(d.t2d_proxy==1),np.nan)
  d['complete']=d.joint.notna()
  d['case']=d.joint.eq(1).astype(int)
  d['eligible_design']=d.WTMEC2YR.gt(0)&d.SDMVSTRA.notna()&d.SDMVPSU.notna()
  d['age_group']=pd.cut(d.RIDAGEYR,[20,45,65,75,np.inf],right=False,labels=['20-44','45-64','65-74','75+'])
  d['sex']=d.RIAGENDR.map({1:'Male',2:'Female'})
  d['race_ethnicity_code']=d.RIDRETH3.astype('Int64').astype(str)
  d['cycle']=label
  return d
cohorts={}
for label,tables in cycles.items():
 try:
  cohorts[label]=harmonize(label,tables)
  print(label,'adults',len(cohorts[label]),'complete',int((cohorts[label].complete&cohorts[label].eligible_design).sum()),'cases',int((cohorts[label].complete&cohorts[label].eligible_design&cohorts[label].case.eq(1)).sum()))
 except Exception as exc:
  errors[label]=str(exc);print('HARMONIZATION FAILED:',label,str(exc))
assert '2017-2018' in cohorts

## 4 — Survey-design Taylor estimates
PSU contributions are formed on the **entire adult MEC design domain**. Incomplete outcomes are zero for case and complete-case denominator totals, preserving design strata and PSU structure. Ratios use Taylor linearization of `case − p × complete`. Two-sided 95% CIs use the design degrees of freedom. This is **not** an independent R cross-check; V3.1 performed that cross-check for the reference cycle.

In [ ]:
def survey_total_var(df,quantity):
    x=df[['SDMVSTRA','SDMVPSU']].copy()
    x['v']=df.WTMEC2YR.to_numpy()*np.asarray(quantity,dtype=float)
    sums=x.groupby(['SDMVSTRA','SDMVPSU'],observed=True).v.sum()
    variance=0.
    for _,g in sums.groupby(level=0):
        a=g.to_numpy();m=len(a)
        if m>1:variance+=m/(m-1)*np.square(a-a.mean()).sum()
    return float(variance)
def survey_stats(d,case_mask,denom_mask):
    z=d.loc[d.eligible_design].copy()
    c=np.asarray(case_mask.reindex(z.index).fillna(False),dtype=float)
    den=np.asarray(denom_mask.reindex(z.index).fillna(False),dtype=float)
    n=float((z.WTMEC2YR*c).sum());tot=float((z.WTMEC2YR*den).sum())
    strata=z.groupby('SDMVSTRA',observed=True).SDMVPSU.nunique()
    dfree=int(np.maximum(strata-1,0).sum())
    if not tot or not dfree:raise ValueError('Empty denominator or invalid survey design')
    p=n/tot;vc=survey_total_var(z,c);vr=survey_total_var(z,c-p*den)
    tcrit=student_t.ppf(.975,dfree)
    return {'weighted_cases':n,'weighted_denominator':tot,'prevalence':p,'count_variance':vc,'prevalence_variance':vr/tot**2,'count_ci_low':max(0,n-tcrit*np.sqrt(vc)),'count_ci_high':n+tcrit*np.sqrt(vc),'prevalence_ci_low':max(0,p-tcrit*np.sqrt(vr)/tot),'prevalence_ci_high':min(1,p+tcrit*np.sqrt(vr)/tot),'design_df':dfree,'unweighted_cases':int(c.sum()),'unweighted_denominator':int(den.sum())}
rows=[]
for label,d in cohorts.items():
  complete=d.complete&d.eligible_design
  case=complete&d.case.eq(1)
  s=survey_stats(d,case,complete)
  rows.append({'cycle':label,'method':'complete_case',**s})
  if label=='2017-2018':
    print('V3.1 comparability:',s['weighted_cases'],'(reference ≈ 9,497,620; deviations require investigation)')
main=pd.DataFrame(rows)
main.to_csv(ROOT/'v4_population_estimates.csv',index=False)
display(main[['cycle','weighted_cases','prevalence','count_ci_low','count_ci_high','unweighted_cases','unweighted_denominator']])

## 5 — Missingness adjustment (explicit sensitivity analysis)
**Post-stratified response propensity adjustment:** within each cycle, form age × sex × race/ethnicity cells. Reweight complete cases by the ratio of all MEC-weighted adults to complete-case adults in the cell. Sparse or empty cells are pooled using progressively coarser grouping; positivity failures are reported. **Do not interpret this as a validated missing-at-random correction.** The adjusted count and rate have descriptive meaning, but the variance calculation here **does not incorporate uncertainty from estimating response propensities**; therefore no adjusted confidence interval is presented. A fully calibrated design-based or multiple-imputation analysis is a future validation step.

In [ ]:
adjusted=[];missingness=[]
for label,d in cohorts.items():
 z=d.loc[d.eligible_design].copy()
 z['responded']=z.complete.astype(int)
 z['stratum_cell']=z.age_group.astype(str)+'|'+z.sex.astype(str)+'|'+z.race_ethnicity_code
 for group in ['age_group','sex','race_ethnicity_code']:
  for value,sub in z.groupby(group,observed=True,dropna=False):
   all_w=sub.WTMEC2YR.sum();observed_w=sub.loc[sub.complete,'WTMEC2YR'].sum()
   missingness.append({'cycle':label,'dimension':group,'group':str(value),'n':len(sub),'weighted_missing_fraction':float(1-observed_w/all_w) if all_w else np.nan})
 # Coarsen empty/small cells using age x sex, then overall (document fallback counts).
 groupcols=[['age_group','sex','race_ethnicity_code'],['age_group','sex'],[]]
 response=np.full(len(z),np.nan)
 fallback=np.full(len(z),-1)
 for level,cols in enumerate(groupcols):
  if cols:
   g=z.groupby(cols,observed=True,dropna=False)
   totals=g.WTMEC2YR.transform('sum')
   # Map group totals using per-row keys to avoid categorical indexing quirks.
   obs_w=g.WTMEC2YR.transform(lambda s: float((s*z.loc[s.index,'complete']).sum()))
   n_obs=g.complete.transform('sum')
  else:
   totals=pd.Series(z.WTMEC2YR.sum(),index=z.index)
   obs_w=pd.Series(z.loc[z.complete,'WTMEC2YR'].sum(),index=z.index)
   n_obs=pd.Series(z.complete.sum(),index=z.index)
  good=(n_obs>=10)&(obs_w>0)&(totals>0)
  fill=np.isnan(response)&good.to_numpy()
  response[fill]=(obs_w.to_numpy()[fill]/totals.to_numpy()[fill])
  fallback[fill]=level
 if np.isnan(response).any():raise ValueError('Unresolved response-propensity cells')
 z['response_probability']=response
 z['adjusted_weight']=np.where(z.complete,z.WTMEC2YR/response,0.)
 pop=float((z.adjusted_weight*z.case).sum())
 allpop=float(z.WTMEC2YR.sum())
 adjusted.append({'cycle':label,'method':'cell_response_adjustment','weighted_cases':pop,'weighted_denominator':allpop,'prevalence':pop/allpop,'min_response_probability':float(response.min()),'max_weight_multiplier':float((1/response).max()),'n_fallback_to_age_sex':int((fallback==1).sum()),'n_fallback_to_overall':int((fallback==2).sum()),'variance_status':'NOT ESTIMATED: propensity uncertainty not incorporated'})
 pd.DataFrame({'SEQN':z.SEQN,'cycle':label,'response_probability':response,'fallback_level':fallback}).to_csv(ROOT/f'v4_response_diagnostics_{label}.csv',index=False)
adjusted_df=pd.DataFrame(adjusted)
adjusted_df.to_csv(ROOT/'v4_missingness_adjusted_sensitivity.csv',index=False)
pd.DataFrame(missingness).to_csv(ROOT/'v4_missingness_by_demographics.csv',index=False)
display(adjusted_df[['cycle','weighted_cases','prevalence','min_response_probability','max_weight_multiplier']])

## 6 — Segment estimates and reliability flags
Segment denominators are the **complete cases within that demographic subgroup**, not all adults. Small counts (`n < 30`) are flagged as unstable; subgroup estimates are descriptive and should not be used to establish treatment market size.

In [ ]:
segments=[]
for label,d in cohorts.items():
 for dim in ['age_group','sex','race_ethnicity_code']:
  for val in d[dim].dropna().unique():
   sub=d[dim].eq(val)&d.complete&d.eligible_design
   if int(sub.sum())==0:continue
   s=survey_stats(d,sub&d.case.eq(1),sub)
   segments.append({'cycle':label,'dimension':dim,'group':str(val),'reliability_flag':'UNSTABLE_N_LT_30' if s['unweighted_cases']<30 else 'REVIEW_CI',**s})
segments_df=pd.DataFrame(segments)
segments_df.to_csv(ROOT/'v4_population_segments.csv',index=False)
display(segments_df[['cycle','dimension','group','weighted_cases','unweighted_cases','reliability_flag']].head(20))

## 7 — Preserve the baseline and export provenance-linked observations
This generates a **separate JSON observation layer**, not additions to the production KG. Each observation contains source identifiers, population definition, estimator, uncertainty, and cautions. No automatic `add_node`/`add_edge` calls are made.

In [ ]:
source_map={label:[x for x in manifest if x['cycle']==label] for label in cohorts}
observations=[]
for _,row in main.iterrows():
 label=row['cycle']
 observations.append({'id':'nhanes:'+label+':ckd_t2d_proxy:complete_case','type':'PopulationObservation','cycle':label,
  'population':'US noninstitutionalized adults age >=20','definition':'single-visit eGFR <60 OR UACR >=30; self-reported diabetes with reported diagnosis age >=30',
  'estimated_count':float(row.weighted_cases),'count_ci_95':[float(row.count_ci_low),float(row.count_ci_high)],
  'prevalence_complete_case':float(row.prevalence),'design_df':int(row.design_df),
  'unweighted_positive':int(row.unweighted_cases),'unweighted_complete':int(row.unweighted_denominator),
  'sources':source_map[label],'baseline_kg_sha256':KG_SHA256,
  'limitations':['not CKD chronicity confirmed','not verified type 2 diabetes','not Kerendia eligibility','missingness may bias estimate','assay comparability and survey guidance require review']})
(ROOT/'v4_population_observations.json').write_text(json.dumps(observations,indent=2))
assert hashlib.sha256(kg_path.read_bytes()).hexdigest()==KG_SHA256
print('Baseline unchanged:',KG_SHA256,'Observations:',len(observations))

## 8 — Validation gate and export
V4 is a **development candidate** until CDC cycle comparability, independent survey checks for all cycles, and missing-data sensitivity are reviewed. The 2017–2018 result must be compared with the original V3.1 output; no hard-coded agreement is assumed. The baseline graph is unchanged.

In [ ]:
ref=main.loc[main.cycle.eq('2017-2018')].iloc[0]
reference_value=9497620.0
relative_deviation=abs(ref.weighted_cases-reference_value)/reference_value
report={'version':'4.0','created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'baseline_kg_sha256':KG_SHA256,'baseline_unchanged':hashlib.sha256(kg_path.read_bytes()).hexdigest()==KG_SHA256,
 'github_accessed':False,'cycles_completed':list(cohorts),'cycle_errors':errors,
 'v3_1_reference_count':reference_value,'v4_reference_count':float(ref.weighted_cases),
 'reference_relative_deviation':relative_deviation,'reference_within_0_1pct':bool(relative_deviation<.001),
 'independent_r_validation_for_v4':'NOT RUN; V3.1 R validation only',
 'adjusted_variance':'NOT ESTIMATED; sensitivity estimates only',
 'gate':'REVIEW_REQUIRED','notes':['No 2019-2020 special-cycle pooling','Do not infer trends without checking lab comparability','Do not interpret as treatment eligibility']}
(ROOT/'v4_validation_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
assert report['baseline_unchanged']
if not report['reference_within_0_1pct']:
 print('WARNING: V3.1 reference differs by >0.1%; investigate before interpreting V4.')
with zipfile.ZipFile(ROOT/'Healthcare_KG_Population_Intelligence_V4_Outputs.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
 for p in ROOT.iterdir():
  if p.name.startswith('v4_') and p.suffix in ('.csv','.json'):archive.write(p,arcname=p.name)
 archive.write(kg_path,arcname=kg_path.name)
print('OUTPUT ZIP:',ROOT/'Healthcare_KG_Population_Intelligence_V4_Outputs.zip')

## 9 — Save to Google Drive / download
This is the **only optional external write** and targets a new development folder. It does not use GitHub.

In [ ]:
from google.colab import files
files.download(str(ROOT/'Healthcare_KG_Population_Intelligence_V4_Outputs.zip'))
# Optional Drive copy (uncomment):
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# dest=Path('/content/drive/MyDrive/Healthcare_KG_Population_Intelligence_V4')
# dest.mkdir(parents=True,exist_ok=True)
# shutil.copy2(ROOT/'Healthcare_KG_Population_Intelligence_V4_Outputs.zip',dest)

### Next validation gate
Review the `v4_validation_report.json`, `v4_population_estimates.csv`, and missingness sensitivity results. If the newer-cycle harmonization fails, inspect CDC documentation rather than renaming variables speculatively. Only after these checks should population observations be integrated into a **copy** of the KG or used for treatment-opportunity modeling.